# Night ECOSTRESS — rings and point control, all night scenes (Phase 7a)

For every night ECOSTRESS scene (ECO_L2T_LSTE, 70 m, mid-2018 onward) over the four paper sites, this
notebook records the cloud-free mean LST in the same five rings as the daytime census, plus the paper's
point-control values (at the verified campus point, at the paper's original point, and at the control).
It reads only a ~12 km window from each file over the internet, so nothing large is downloaded.

**Order:** Part 0 (log in) → Part 1 (find scenes) → Part 2 (extract; ~30–60 min; safe to re-run, it
skips scenes already done) → Part 3 (check). Results go to `census/outputs/night/` on Drive; the
statistics are run by Claude locally.

## Part 0 — Setup and NASA Earthdata login

In [ ]:
!pip install -q earthaccess rioxarray

from google.colab import drive
drive.mount("/content/drive")

import os, sys, importlib
import numpy as np, pandas as pd
import earthaccess, rasterio
from rasterio.windows import from_bounds, Window
from pyproj import Transformer
from concurrent.futures import ThreadPoolExecutor

BASE = "/content/drive/MyDrive/2026/Remote-Sensing/census"
sys.path.insert(0, f"{BASE}/code")
import dc_heat.night
importlib.reload(dc_heat.night)
from dc_heat import night

OUT = f"{BASE}/outputs/night"
os.makedirs(OUT, exist_ok=True)

earthaccess.login(persist=True)          # asks for your Earthdata username/password once; saves ~/.netrc
os.environ.update({"GDAL_HTTP_COOKIEFILE": "/root/cookies.txt", "GDAL_HTTP_COOKIEJAR": "/root/cookies.txt",
                   "GDAL_DISABLE_READDIR_ON_OPEN": "EMPTY_DIR", "CPL_VSIL_CURL_ALLOWED_EXTENSIONS": "TIF",
                   "GDAL_HTTP_MAX_RETRY": "5", "GDAL_HTTP_RETRY_DELAY": "2"})

RINGS = [(0.0, 0.5), (0.5, 1.0), (1.0, 2.0), (2.0, 3.5), (3.5, 5.0)]   # km, same as the daytime census
# verified = your checked campus point; paper = the point used for Table V; control = the paper's control.
SITES = pd.DataFrame([
    dict(site_id="MESA-META", group="mesa", lat=33.34740331, lon=-111.6322664, paper_lat=33.3472, paper_lon=-111.6325,
         control_lat=33.3472, control_lon=-111.5787),
    dict(site_id="MESA-EDGECORE", group="mesa", lat=33.346345, lon=-111.628149, paper_lat=33.346345, paper_lon=-111.628149,
         control_lat=33.3463, control_lon=-111.5751),
    dict(site_id="MIDLOTHIAN", group="midlothian", lat=32.44312881, lon=-97.06241477, paper_lat=32.443078, paper_lon=-97.048766,
         control_lat=32.462586, control_lon=-97.094530),
    dict(site_id="GALLATIN", group="gallatin", lat=36.41209892, lon=-86.36961485, paper_lat=36.411583, paper_lon=-86.368952,
         control_lat=36.40407, control_lon=-86.32540),
])
print(SITES[["site_id", "group", "lat", "lon"]].to_string(index=False))

## Part 1 — Find every night scene (a minute or two)

In [ ]:
granules = {}
for group, sites in SITES.groupby("group"):
    first = sites.iloc[0]
    found = earthaccess.search_data(short_name="ECO_L2T_LSTE", temporal=("2018-07-01", "2026-12-31"),
                                    point=(first.lon, first.lat))
    found = [g for g in found if g["umm"].get("DataGranule", {}).get("DayNightFlag") == "Night"]
    granules[group] = found
    years = pd.Series([g["umm"]["TemporalExtent"]["RangeDateTime"]["BeginningDateTime"][:4] for g in found])
    print(f"{group:11s} {len(found):4d} night scenes  by year: {years.value_counts().sort_index().to_dict()}")

## Part 2 — Extract rings and points (about 30–60 min; re-run to resume)

Each scene is read in a window around the site group, cloud pixels are removed using the scene's
cloud layer, and the ring and point values are appended to `night_scenes_<group>.csv`.

In [ ]:
HALF_M = 6000          # window half-width around the group centre, metres (rings reach 5 km)

def links(granule):
    urls = granule.data_links()
    pick = lambda suffix: next((u for u in urls if u.endswith(suffix)), None)
    return pick("_LST.tif"), pick("_cloud.tif")

def read_window(url, lon, lat):
    with rasterio.open(url) as src:
        to_xy = Transformer.from_crs("EPSG:4326", src.crs, always_xy=True)
        x, y = to_xy.transform(lon, lat)
        win = from_bounds(x - HALF_M, y - HALF_M, x + HALF_M, y + HALF_M, src.transform)
        win = win.round_offsets().round_lengths().intersection(Window(0, 0, src.width, src.height))
        if win.width < 2 or win.height < 2:
            raise ValueError("site window falls outside this tile")
        data = src.read(1, window=win).astype(float)
        t = src.window_transform(win)
        xs = t.c + t.a * (np.arange(data.shape[1]) + 0.5)
        ys = t.f + t.e * (np.arange(data.shape[0]) + 0.5)
        return data, xs, ys, to_xy

def process(group, sites, granule):
    meta = granule["umm"]
    gid = meta["GranuleUR"]
    utc = pd.Timestamp(meta["TemporalExtent"]["RangeDateTime"]["BeginningDateTime"]).tz_localize(None)
    lst_url, cloud_url = links(granule)
    centre_lon, centre_lat = sites.lon.mean(), sites.lat.mean()
    try:
        raw, xs, ys, to_xy = read_window(lst_url, centre_lon, centre_lat)
    except Exception as err:            # e.g. the window falls outside this tile
        return [], f"{gid}: {err}"
    values = night.to_celsius(raw)
    cloud_used = False
    if cloud_url:
        try:
            cloud, _, _, _ = read_window(cloud_url, centre_lon, centre_lat)
            values[cloud != 0] = np.nan
            cloud_used = True
        except Exception:
            pass
    rows = []
    for s in sites.itertuples():
        base = {"site_id": s.site_id, "granule": gid, "utc": utc,
                "local_hour": night.local_solar_hour(utc, s.lon), "cloud_mask": cloud_used}
        cx, cy = to_xy.transform(s.lon, s.lat)
        for i, ring in enumerate(night.ring_means(values, xs, ys, cx, cy, RINGS)):
            rows.append({**base, "zone": f"ring{i}", "lst_c": ring["mean"], "valid_frac": ring["valid_frac"]})
        for zone, lon, lat in (("pt_verified", s.lon, s.lat), ("pt_paper", s.paper_lon, s.paper_lat),
                               ("pt_control", s.control_lon, s.control_lat)):
            px, py = to_xy.transform(lon, lat)
            rows.append({**base, "zone": zone, "lst_c": night.point_mean(values, xs, ys, px, py), "valid_frac": np.nan})
    return rows, None

for group, sites in SITES.groupby("group"):
    path = f"{OUT}/night_scenes_{group}.csv"
    done = set(pd.read_csv(path).granule) if os.path.exists(path) else set()
    todo = [g for g in granules[group] if g["umm"]["GranuleUR"] not in done]
    print(f"{group}: {len(done)} scenes already done, {len(todo)} to go")
    errors = []
    for start in range(0, len(todo), 40):                     # save every 40 scenes
        batch = todo[start:start + 40]
        with ThreadPoolExecutor(max_workers=8) as pool:
            results = list(pool.map(lambda g: process(group, sites, g), batch))
        rows = [r for rs, _ in results for r in rs]
        errors += [e for _, e in results if e]
        if rows:
            pd.DataFrame(rows).to_csv(path, mode="a", header=not os.path.exists(path), index=False)
        print(f"  {group}: {min(start + 40, len(todo))}/{len(todo)}")
    if errors:
        pd.DataFrame({"error": errors}).to_csv(f"{OUT}/night_errors_{group}.csv", index=False)
        print(f"  {len(errors)} scenes could not be read (listed in night_errors_{group}.csv); first: {errors[0][:200]}")

## Part 3 — Check: usable scenes per site and year (core and background ring at least half clear)

In [ ]:
for group in SITES.group.unique():
    df = pd.read_csv(f"{OUT}/night_scenes_{group}.csv", parse_dates=["utc"])
    rings = df[df.zone.isin(["ring0", "ring4"])].pivot_table(index=["site_id", "granule", "utc"], columns="zone",
                                                              values="valid_frac").reset_index()
    usable = rings[(rings.ring0 >= 0.5) & (rings.ring4 >= 0.5)]
    print(f"\n{group}: cloud mask applied in {df.cloud_mask.mean():.0%} of rows")
    print(usable.groupby(["site_id", usable.utc.dt.year]).size().unstack(fill_value=0).to_string())